In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rc
import folium
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import random
from datetime import datetime
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import sys
modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'folium': 'folium',
    'beautifulsoup4': 'bs4',
    'selenium': 'selenium',
    'lxml': 'lxml'
}
print("=" * 50)
print("모듈 설치 확인")
print("=" * 50)
for package, import_name in modules.items():
    try:
        module = __import__(import_name)
        version = getattr(module, '__version__', 'unknown')
        print(f"✓ {package:20s} {version}")
    except ImportError:
        print(f"✗ {package:20s} 설치 안됨")
print("=" * 50)

모듈 설치 확인
✓ numpy                2.3.5
✓ pandas               2.3.3
✓ matplotlib           3.10.8
✓ seaborn              0.13.2
✓ folium               0.20.0
✓ beautifulsoup4       4.14.3
✓ selenium             4.39.0
✓ lxml                 6.0.2


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 1. 데이터 로드 (한글 깨짐 방지 옵션 추가)
df = pd.read_csv('통합자료.csv', encoding='cp949')

# [추가] 숫자 데이터에 포함된 쉼표(,) 제거 및 숫자형 변환
# 엑셀 데이터 특성상 "3,027" 처럼 쉼표가 있으면 숫자로 인식을 못합니다.
for col in ['설비용량(KW)', '1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월', '9월', '10월', '11월', '12월']:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '').astype(float)

# 2. 발전량 총합 계산 (종속변수 y 만들기)
# 파일에 월별 데이터만 있으므로, 이를 합쳐서 연간 총 발전량을 만듭니다.
month_cols = ['1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월', '9월', '10월', '11월', '12월']
df['연간총발전량'] = df[month_cols].sum(axis=1)

# 3. 데이터 전처리 (독립변수 X 설정)
# 현재 파일에서 확실히 쓸 수 있는 데이터는 '연도'와 '설비용량(KW)'입니다.
X = df[['연도', '설비용량(KW)']]
y = df['연간총발전량']

# 4. 모델 학습
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

print("학습 완료! 모델이 데이터의 패턴을 파악했습니다.")

# 5. 2026년 예측 (예시: 경기도의 2024년 설비용량이 유지된다고 가정할 때)
# 2024년 경기도 설비용량 약 3,027,000 KW 가정
future_2026 = pd.DataFrame({'연도': [2026], '설비용량(KW)': [3027000]})
pred_2026 = model.predict(future_2026)

print(f"\n--- 2026년 예측 결과 ---")
print(f"2026년 예상 연간 총 발전량: {pred_2026[0]:,.2f} MWh")

학습 완료! 모델이 데이터의 패턴을 파악했습니다.

--- 2026년 예측 결과 ---
2026년 예상 연간 총 발전량: 24,790,930.61 MWh


In [8]:
# 모델의 계수(Coefficients) 확인
coef_df = pd.DataFrame({
    '변수': X.columns,
    '영향력(계수)': model.coef_
})

print("--- 변수별 영향력 분석 ---")
print(coef_df.sort_values(by='영향력(계수)', ascending=False))

--- 변수별 영향력 분석 ---
         변수        영향력(계수)
0        연도  298574.398561
1  설비용량(KW)       7.802668


In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 1. 데이터 로드 및 전처리
df = pd.read_csv('통합자료.csv', encoding='cp949')

# 숫자 데이터 전처리 (쉼표 제거 및 숫자형 변환)
cols_to_fix = ['설비용량(KW)', '1월', '2월', '3월', '4월', '5월', '6월', 
               '7월', '8월', '9월', '10월', '11월', '12월']

for col in cols_to_fix:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '').astype(float)

# 연간 총 발전량 계산
month_cols = ['1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월', '9월', '10월', '11월', '12월']
df['연간총발전량'] = df[month_cols].sum(axis=1)

# 2. 지역별 예측 함수 정의
def predict_2026_by_region(data):
    results = []
    regions = data['구분'].unique()
    
    for region in regions:
        # 해당 지역 데이터만 필터링
        region_df = data[data['구분'] == region].sort_values('연도')
        
        # 학습 데이터 준비 (X: 연도, 설비용량 / y: 발전량)
        X = region_df[['연도', '설비용량(KW)']]
        y = region_df['연간총발전량']
        
        # 모델 생성 및 학습
        model = LinearRegression()
        model.fit(X, y)
        
        # 2026년 시나리오 설정
        # 설비용량 예측: 최근 2개년 평균 성장률 적용 (예: 연평균 10% 증가 가정)
        last_capacity = region_df.iloc[-1]['설비용량(KW)']
        est_capacity_2026 = last_capacity * (1.10 ** 2) # 2년치 복리 성장
        
        future_X = pd.DataFrame({
            '연도': [2026],
            '설비용량(KW)': [est_capacity_2026]
        })
        
        # 발전량 예측
        pred_gen = model.predict(future_X)[0]
        
        # 이용률 계산 (발전량 / (용량 * 8760)) * 100
        util_rate = (pred_gen * 1000) / (est_capacity_2026 * 8760) * 100
        
        results.append({
            '지역': region,
            '2026_예상발전량(MWh)': round(pred_gen, 2),
            '2026_예상이용률(%)': round(util_rate, 2),
            '2026_예상설비용량(KW)': round(est_capacity_2026, 0)
        })
    
    return pd.DataFrame(results)

# 3. 결과 출력
final_forecast = predict_2026_by_region(df)
print(final_forecast.sort_values(by='2026_예상발전량(MWh)', ascending=False))

    지역  2026_예상발전량(MWh)  2026_예상이용률(%)  2026_예상설비용량(KW)
11  전남       8729609.93         135.06         737837.0
16  전북       6673111.00         220.28         345816.0
12  경북       5297118.35          96.47         626826.0
10  충남       5232869.12          70.65         845556.0
13  경남       2683408.48          93.26         328475.0
8   경기       2586979.26          71.81         411267.0
15  강원       2462918.32         127.39         220703.0
9   충북       2112537.25          97.01         248582.0
2   대구        562023.49          65.59          97820.0
14  제주        559043.04         480.70          13276.0
4   광주        423758.49          80.42          60153.0
1   부산        366583.38          84.61          49461.0
6   울산        182972.24          57.39          36394.0
3   인천        180896.12          69.91          29539.0
7   세종        103528.54          93.12          12692.0
5   대전         81972.78          73.88          12666.0
0   서울         64936.75        2172.46          

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# 1. 각각의 파일 불러오기
df_main = pd.read_csv('통합자료.csv', encoding='cp949')
df_temp = pd.read_csv('평균 기온.csv', encoding='cp949')
df_sun = pd.read_csv('일사량(연도와 상관없이 일정함)/한국동서발전(주)_대기권 밖 일사량_20240125.csv', encoding='cp949')

# [전처리] '통합자료'에서 연간총발전량 계산 및 쉼표 제거
cols_to_fix = ['설비용량(KW)', '1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월', '9월', '10월', '11월', '12월']
for col in cols_to_fix:
    if df_main[col].dtype == 'object':
        df_main[col] = df_main[col].str.replace(',', '').astype(float)
        
month_cols = ['1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월', '9월', '10월', '11월', '12월']
df_main['연간총발전량'] = df_main[month_cols].sum(axis=1)

# [데이터 합치기] '구분(지역)'과 '연도'를 기준으로 합칩니다.
# 주의: 파일마다 지역명(서울, 서울특별시 등)이나 연도 형식이 같아야 합니다.
df = pd.merge(df_main, df_temp, on=['연도', '구분'], how='inner')

# 일사량 데이터는 연도와 상관없으므로 '지역' 기준으로만 합침 (df_sun 전처리가 필요할 수 있음)
# 여기서는 예시로 df_main과 df_temp만 합친 데이터로 진행합니다.
# 일사량까지 합치려면 df_sun의 '지역'명을 '구분'으로 맞춰주는 작업이 선행되어야 합니다.

# 2. 독립변수(X)와 종속변수(y) 설정 (실제 파일에 적힌 컬럼명으로 수정)
# 컬럼명에 공백이나 특수문자가 있는지 print(df.columns)로 확인해보세요.
X = df[['설비용량(KW)', '평균기온[℃]', '연도']] 
y = df['연간총발전량']

# 3. 랜덤 포레스트 모델 학습
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# 4. 시각화 (동일)
importances = rf_model.feature_importances_
feature_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_df['Feature'], feature_df['Importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title('발전량 결정 요소 중요도')
plt.show()

print(feature_df)

KeyError: '구분'